# Full E2E Train: RepViT-M1.5 + T5-Efficient-Mini on Kaggle T4 x2

This notebook runs full end-to-end raw image training: RepViT is inside the graph and is trained in both align and finetune. It does not run the feature cache command.


In [ ]:
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"
USE_SAFE_CONFIG = False

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["USE_SAFE_CONFIG"] = "1" if USE_SAFE_CONFIG else "0"
print({"repo": GITHUB_REPO_URL, "branch": GITHUB_BRANCH, "safe_config": USE_SAFE_CONFIG})

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving/.git ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
else
  cd Efficient_VLM_For_Autonomous_Driving
  git fetch origin "$GITHUB_BRANCH"
  git checkout "$GITHUB_BRANCH"
  git pull --ff-only origin "$GITHUB_BRANCH"
fi
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
python -m pip install -q -e .


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as exc:
    print(f"HF_TOKEN not loaded: {exc}")


In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end
fi
echo "CONFIG=$CONFIG"
echo "PROFILE=$PROFILE"
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full --debug --debug-samples 2


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe.yaml; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end.yaml; fi
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage align --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end
fi
RESUME=""
if [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then
  RESUME="--resume $PROFILE/checkpoints/align_latest.pt"
fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align $RESUME --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 50


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe; else PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end; fi
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
python -m efficient_vlm_ad monitor-progress --output-dir "$PROFILE" --last 5 || true
nvidia-smi || true


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end
fi
if [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then
  RESUME="$PROFILE/checkpoints/finetune_latest.pt"
else
  RESUME="$PROFILE/checkpoints/align_best.pt"
fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$RESUME" --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 50


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe; else PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end; fi
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
python -m efficient_vlm_ad monitor-progress --output-dir "$PROFILE" --last 5 || true
ls -lh "$PROFILE/checkpoints" || true


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_end2end.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end
fi
CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
if [ ! -f "$CHECKPOINT" ]; then
  CHECKPOINT="$PROFILE/checkpoints/finetune_latest.pt"
fi
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT" --batch-size 16 --debug --debug-samples 1
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT" --debug --debug-samples 1
tail -n 20 "$PROFILE/debug/eval_progress.jsonl" || true
tail -n 20 "$PROFILE/debug/benchmark_progress.jsonl" || true


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end_safe; else PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_end2end; fi
ls -lh "$PROFILE/checkpoints" || true
cat "$PROFILE/metrics.json" || true
cat "$PROFILE/benchmark.json" || true
tail -n 20 "$PROFILE/debug/eval_progress.jsonl" || true
tail -n 20 "$PROFILE/debug/benchmark_progress.jsonl" || true
